# Drawing Tree Grids

When using `toytree.MultiTree` objects you can draw multiple trees as a grid on a shared canvas with `MultiTree.draw()`. This page keeps the same example-driven workflow as the older multitree docs, but updates the wording and code to the current API.

In [ ]:
import toytree

### Example dataset

In [ ]:
# a MultiTree containing 50 random coalescent trees with 10 tips each
mtree = toytree.mtree([toytree.rtree.coaltree(10, seed=i) for i in range(50)])

In [ ]:
mtree.draw();

In [ ]:
# a multi-newick string
NEWICKS = """\
(((a:1,b:1):1,(d:1.5,e:1.5):0.5):1,c:3);
(((a:1,d:1):1,(b:1,e:1):1):1,c:3);
(((a:1.5,b:1.5):1,(d:1,e:1):1.5):1,c:3.5);
(((a:1.25,b:1.25):0.75,(d:1,e:1):1):1,c:3);
(((a:1,b:1):1,(d:1.5,e:1.5):0.5):1,c:3);
(((b:1,a:1):1,(d:1.5,e:1.5):0.5):2,c:4);
(((a:1.5,b:1.5):0.5,(d:1,e:1):1):1,c:3);
(((b:1.5,d:1.5):0.5,(a:1,e:1):1):1,c:3);
"""

In [ ]:
# create a multitree object
mtree = toytree.mtree(NEWICKS)

### Grid tree drawings

The `.draw()` function accepts the same styling arguments as `ToyTree.draw()`, but adds a few grid-specific arguments for organizing multiple trees on one canvas. The most useful are `shape`, `shared_axes`, `idxs`, `margin`, `fixed_order`, and `label`.

`MultiTree.draw()` returns three objects, similar to `ToyTree.draw()`, but instead of `(Canvas, Cartesian, Mark)` it returns `(Canvas, list[Cartesian], list[Mark])`. The axes list includes one entry per grid cell, while the marks list includes one entry per tree that was actually rendered.

As in the older multitree docs, the examples below keep the trees visible enough to compare directly, including tip labels in most drawings.

In [ ]:
mtree.draw();

**shape** -- The `shape` argument arranges trees on the canvas as `(rows, columns)`. This is usually the first setting to adjust when you want a denser comparison panel or more room for tip labels.

In [ ]:
mtree.draw(shape=(2, 2), width=400, height=400);

**shared_axes** -- The `shared_axes` argument can be used on linear layouts such as `"r"`, `"l"`, `"u"`, and `"d"` to place all trees on the same depth scale. This is useful for comparing node heights or branch lengths among trees.

In [ ]:
mtree.draw(scale_bar=True, layout='d', shared_axes=True);

**margin** -- You can set the margin size in pixel units between trees in the grid. Larger margins make room for long labels, while smaller margins pack trees more tightly. Enter either one value for all four sides or a tuple `(top, right, bottom, left)`.

In [ ]:
mtree.draw(margin=(10, 30, 10, 30));

**style and label axes** -- Because `MultiTree.draw()` returns the subplot axes, you can still style scales, ticks, labels, and titles after the trees are drawn.

In [ ]:
# draw a tree grid while using shared_axes=True to share the time axis
canvas, axes, marks = mtree.draw(
    layout='d',
    edge_type='c',
    node_sizes=5,
    node_mask=False,
    shape=(1, 4),
    width=650,
    height=250,
    scale_bar=True,
    shared_axes=True,
);

# add a label to each subplot
for adx, ax in enumerate(axes):
    ax.label.text = f'tree {adx}'

# add a y-axis label to only the first subplot
axes[0].y.label.text = '        time (generations)'
axes[0].y.label.offset = 25

canvas

### Tree grid styling

All standard `ToyTree.draw()` styling arguments can be passed to `MultiTree.draw()` to apply one style to every tree in the grid. You can also prepare tree-specific style features on each `ToyTree` before building the `MultiTree`, and then map those features during the shared draw call.

In [ ]:
# build a small tree set with the same tip names across trees
names = [
    'Priapella',
    'Psjonesii',
    'Xalvarezi',
    'Xmayae',
    'Xhellerii',
    'Xsignum',
    'Xandersi',
    'Xcouchianus',
]

fish = toytree.mtree([
    toytree.rtree.unittree(ntips=8, names=names, seed=i)
    for i in range(4)
])

In [ ]:
fish.draw(shape=(1, 4), height=320);

For example, here we assign one feature value per tree and use it to color the tip labels and edges differently in each panel. This replaces the older `tree.style` example, which is no longer part of the public API.

In [ ]:
styled = []
for idx, tree in enumerate(fish.treelist):
    styled_tree = (
        tree
        .set_node_data(
            'tip_group',
            {node.name: idx for node in tree[:tree.ntips]},
            default=idx,
        )
        .set_node_data(
            'edge_group',
            {node.idx: idx for node in tree[tree.ntips:]},
            default=idx,
            edge=True,
        )
    )
    styled.append(styled_tree)

styled_fish = toytree.mtree(styled)
styled_fish.draw(
    shape=(1, 4),
    height=320,
    width=900,
    tip_labels_align=True,
    tip_labels_colors=('tip_group', 'Set2'),
    edge_colors=('edge_group', 'Set2'),
);

The argument `fixed_order` is especially useful for multitree drawings when you want discordance among trees to stand out. It fixes the order in which tips are plotted so that topological differences appear as visible conflicts. Here we infer a consensus tree from the full set and reuse its tip order explicitly.

In [ ]:
# extend to six trees so the ordering differences are easier to see
fish = toytree.mtree([
    toytree.rtree.unittree(ntips=8, names=names, seed=i)
    for i in range(6)
])

# infer a consensus tree and reuse its tip order
consfish = fish.get_consensus_tree()

fish.draw(
    shape=(2, 3),
    height=600,
    width=760,
    fixed_order=consfish.get_tip_labels(),
    edge_type='c',
    shared_axes=True,
    layout='d',
);

## Related APIs

- `MultiTree.draw_cloud_tree()` overlays many trees on a single set of axes instead of drawing them on a grid.
- `ToyTree.draw()` accepts the shared draw-style arguments that `MultiTree.draw()` forwards to every rendered tree.